# Bài 3 — Lọc Detections như lọc DataFrame

**Mục tiêu:** Thành thạo kỹ thuật lọc — kỹ năng dùng nhiều nhất trong dự án thực tế.

## 0. Chuẩn bị (asset video + display.py dùng chung)

In [1]:
!pip install -q supervision ultralytics "supervision[assets]"

In [2]:
from supervision.assets import download_assets, VideoAssets

download_assets(VideoAssets.VEHICLES)
print(VideoAssets.VEHICLES.value)  # "vehicles.mp4"

[2026-08-18 10:04:22] [INFO] supervision.assets.downloader - vehicles.mp4 asset download complete.
vehicles.mp4


In [4]:
%%writefile display.py
# display.py — hàm hiển thị dùng chung cho toàn giáo trình
import cv2

WINDOW_NAME = "Supervision - Live"
MAX_DISPLAY_WIDTH = 1280   # thu nhỏ frame cho vừa màn hình (chỉ để XEM, không ảnh hưởng xử lý)


def show_frame(frame, window_name: str = WINDOW_NAME, wait: int = 1) -> bool:
    """Hiện frame lên cửa sổ. Trả về False nếu người dùng bấm Q/ESC (muốn thoát).

    wait=1  -> dùng cho video (hiện liên tục, không chặn)
    wait=0  -> dùng cho ảnh tĩnh (dừng lại chờ bấm phím bất kỳ)
    """
    h, w = frame.shape[:2]
    if w > MAX_DISPLAY_WIDTH:                      # thu nhỏ để vừa màn hình
        scale = MAX_DISPLAY_WIDTH / w
        frame = cv2.resize(frame, (int(w * scale), int(h * scale)))

    cv2.imshow(window_name, frame)
    key = cv2.waitKey(wait) & 0xFF
    if key in (ord("q"), ord("Q"), 27):            # Q hoặc ESC -> thoát
        return False
    return True


def close_windows():
    cv2.destroyAllWindows()

Overwriting display.py


## Lấy detections mẫu ( Bài 1)

In [5]:
import cv2
import numpy as np
import supervision as sv
from ultralytics import YOLO

from display import show_frame, close_windows

model = YOLO("yolov8n.pt")
image = next(sv.get_video_frames_generator("vehicles.mp4"))
results = model(image)[0]
detections = sv.Detections.from_ultralytics(results)
print(detections)


0: 384x640 3 cars, 1 truck, 108.6ms
Speed: 3.6ms preprocess, 108.6ms inference, 2.1ms postprocess per image at shape (1, 3, 384, 640)
Detections(xyxy=array([[     2941.1,      1269.3,      3220.8,      1500.7],
       [     944.89,      899.64,      1235.4,      1308.8],
       [     1439.8,      1077.8,      1621.3,      1231.4],
       [     1480.1,      1007.2,      1624.4,      1111.9]], dtype=float32), mask=None, confidence=array([    0.85173,     0.67523,     0.64499,     0.38049], dtype=float32), class_id=array([2, 7, 2, 2]), tracker_id=None, data={'class_name': array(['car', 'truck', 'car', 'car'], dtype='<U5')}, metadata={})


## 3.1. Cú pháp lọc bằng boolean mask

In [6]:
VEHICLE_CLASSES = [2, 3, 5, 7]   # car, motorcycle, bus, truck (COCO)

# Chỉ giữ đối tượng confidence > 0.5
high_conf = detections[detections.confidence > 0.5]

# Chỉ giữ xe hơi (COCO class 2)
only_cars = detections[detections.class_id == 2]

# Giữ nhiều class: car, motorcycle, bus, truck
vehicles = detections[np.isin(detections.class_id, VEHICLE_CLASSES)]

# Kết hợp nhiều điều kiện
combined = detections[
    (detections.confidence > 0.4) & np.isin(detections.class_id, VEHICLE_CLASSES)
]

# Lọc theo diện tích box (loại box quá nhỏ = nhiễu)
big_enough = detections[detections.area > 1000]

print("Tổng ban đầu:", len(detections))
print("Sau lọc kết hợp:", len(combined))

Tổng ban đầu: 4
Sau lọc kết hợp: 3


## 3.2.  Xem trực quan "trước – sau" khi lọc

In [8]:
box_annotator = sv.BoxAnnotator(thickness=2)

before = box_annotator.annotate(image.copy(), detections)               # chưa lọc
filtered = detections[(detections.confidence > 0.4)
                      & np.isin(detections.class_id, VEHICLE_CLASSES)]
after = box_annotator.annotate(image.copy(), filtered)                  # đã lọc

cv2.putText(before, f"TRUOC LOC: {len(detections)}", (20, 60),
            cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 0, 255), 4)
cv2.putText(after, f"SAU LOC: {len(filtered)}", (20, 60),
            cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 255, 0), 4)

side_by_side = np.hstack([before, after])   # ghép ngang 2 ảnh
show_frame(side_by_side, window_name="Truoc vs Sau khi loc", wait=0)
close_windows()

# Thống kê theo class (yêu cầu checkpoint)
names, counts = np.unique(filtered.data["class_name"], return_counts=True)
for name, count in zip(names, counts):
    print(f"{name}: {count}")

car: 2
truck: 1


## 3.3. NMS và các phép xử lý khác

In [9]:
# Non-Max Suppression thủ công (khi model chưa làm hoặc gộp nhiều model)
nms_result = detections.with_nms(threshold=0.5, class_agnostic=False)
print("Trước NMS:", len(detections), "| Sau NMS:", len(nms_result))

# Gộp detections từ 2 model khác nhau (minh họa cú pháp, cần 2 model thật để chạy):
# merged = sv.Detections.merge([detections_yolo, detections_rtdetr]).with_nms(0.5)

Trước NMS: 4 | Sau NMS: 4


## 3.4. Lấy tọa độ điểm neo (anchor)

In [10]:
# Tâm đáy box (điểm "chạm đất" — chuẩn cho xe cộ, người)
points = filtered.get_anchors_coordinates(anchor=sv.Position.BOTTOM_CENTER)
print(points)
# Các anchor khác: CENTER, TOP_LEFT, BOTTOM_RIGHT...

[[       3081      1500.7]
 [     1090.2      1308.8]
 [     1530.5      1231.4]]


##  Checkpoint Bài 3

Cửa sổ hiện 2 ảnh cạnh nhau "TRUOC LOC / SAU LOC" với số lượng box khác nhau rõ rệt; console in thống kê dạng `car: 12, truck: 3`.